# ClassCounter Backend Benchmarks

Compares the performance of each counting backend across different input sizes.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from classcounter import count_classes
from classcounter.core import _count_numba, _count_numpy

## Setup

Define the input sizes to benchmark and a helper that times each call.

In [ ]:
NUM_CLASSES = 20
REPEATS = 20

has_cuda = torch.cuda.is_available()
print(f"CUDA available: {has_cuda}")
if has_cuda:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
SIZES = np.logspace(np.log10(10_000), np.log10(100_000_000), 20, dtype=int)
SIZES

In [ ]:
def bench(fn, sync=False, repeats=REPEATS):
    """Return all wall-clock times in seconds for calling fn().

    Set sync=True for GPU workloads to call torch.cuda.synchronize()
    before and after each timed iteration.
    """
    times = []
    for _ in range(repeats):
        if sync:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if sync:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.array(times)

## Run Benchmarks

In [ ]:
rng = np.random.default_rng(42)

backends = ["numpy", "numba", "torch (cpu)"]
if has_cuda:
    backends.append("torch (gpu)")

results = {b: [] for b in backends}

# Pre-create all inputs so allocation/transfer isn't included in timings
inputs = {}
for size in SIZES:
    arr = rng.integers(0, NUM_CLASSES, size=int(size), dtype=np.uint8)
    tensor_cpu = torch.from_numpy(arr)
    inputs[size] = {
        "numpy": arr,
        "tensor_cpu": tensor_cpu,
        "tensor_gpu": tensor_cpu.cuda() if has_cuda else None,
    }

if has_cuda:
    torch.cuda.synchronize()
print(f"Pre-allocated {len(SIZES)} input sets")

for size in SIZES:
    data = inputs[size]

    results["numpy"].append(bench(lambda: _count_numpy(data["numpy"])))
    results["numba"].append(bench(lambda: _count_numba(data["numpy"])))
    results["torch (cpu)"].append(bench(lambda: count_classes(data["tensor_cpu"])))
    if has_cuda:
        results["torch (gpu)"].append(bench(lambda: count_classes(data["tensor_gpu"]), sync=True))

    print(f"{size:>12,} elements — ", end="")
    print(", ".join(f"{b}: {np.median(results[b][-1]) * 1000:.2f}ms" for b in backends))

## Plot Results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

styles = {
    "numpy": {"marker": "o", "color": "#4C72B0"},
    "numba": {"marker": "s", "color": "#DD8452"},
    "torch (cpu)": {"marker": "^", "color": "#55A868"},
    "torch (gpu)": {"marker": "D", "color": "#C44E52"},
}

for backend in backends:
    all_times = results[backend]  # list of arrays, one per size
    medians = np.array([np.median(t) for t in all_times]) * 1000

    color = styles[backend]["color"]
    ax.plot(SIZES, medians, label=backend, linewidth=2, markersize=7, **styles[backend])

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Number of elements")
ax.set_ylabel("Time (ms)")
ax.set_title("count_classes — Backend Performance — uint8")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()